# **Implement a Deep Reinforcement Learning method for an Acrobot Game**

In [ ]:
import gym
import matplotlib.pyplot as plt
import numpy as np
import math
import random
import matplotlib
import matplotlib.pyplot as plt
from collections import namedtuple, deque
from itertools import count


import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

#optional
import sys
import pickle
import time

In [ ]:
# Create the environment
env = gym.make("Acrobot-v1")

# Set up matplotlib for interactive plotting
# (for use in Jupyter notebooks or IPython)
is_ipython = "inline" in plt.get_backend()
if is_ipython:
    from IPython import display
plt.ion()

# Set up the device to use (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Print the device being used
print(device)

In [ ]:
class DQN(nn.Module):
    """
    Deep Q-Network (DQN) class.

    Parameters:
        n_observations (int): Number of observations (input size).
        n_actions (int): Number of possible actions (output size).
    """

    def __init__(self, n_observations, n_actions):
        super(DQN, self).__init__()
        self.layer1 = nn.Linear(n_observations, 128) # First hidden layer with 128 neurons
        self.layer2 = nn.Linear(128, 64) # Second hidden layer with 128 neurons
        self.layer3 = nn.Linear(64, n_actions) # Output layer with n_actions neurons

    def forward(self, x):
        """
        Forward pass through the network.

        Parameters:
            x (tensor): Input tensor with shape (batch_size, n_observations).

        Returns:
            tensor: Output tensor with shape (batch_size, n_actions).
        """
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        return self.layer3(x)

In [ ]:
class ReplayMemory(object):
    """Replay Memory implementation to store transitions and sample from them"""
    
    def __init__(self, capacity):
        """Initialize the Replay Memory
        
        Args:
        capacity (int): Maximum capacity of the Replay Memory
        """
        self.memory = deque([], maxlen=capacity)

    def push(self, *args):
        """Save a transition
        
        Args:
        *args: Tuple of state, action, next state, and reward
        """
        self.memory.append(Transition(*args))

    def sample(self, batch_size):
        """Sample a random batch of transitions
        
        Args:
        batch_size (int): Number of transitions to sample
        
        Returns:
        List of randomly sampled transitions from the Replay Memory
        """
        return random.sample(self.memory, batch_size)

    def __len__(self):
        """Return the length of the Replay Memory"""
        return len(self.memory)

In [ ]:
def select_action(state):
    """
    Select an action to take based on the current state of the environment

    Args:
    - state (torch.tensor): current state of the environment

    Returns:
    - (torch.tensor): action to take based on the current state
    """
    global eps_threshold
    sample = random.random() # generate a random number between 0 and 1

    if sample > eps_threshold: # if the random number is greater than the threshold, select the action with the highest Q-value
        with torch.no_grad():
            # t.max(1) will return the largest column value of each row.
            # the second column on max result is index of where max element was
            # found, so we pick action with the larger expected reward.
            return policy_net(state).max(1)[1].view(1, 1)
    else: # else, select a random action from the environment's action space
        return torch.tensor([[env.action_space.sample()]], device=device, dtype=torch.long)

In [ ]:
def optimize_model():
    """
    This function optimizes the Q-Network by performing backpropagation and updating the network's parameters. 
    It relies on the global variables defined in the main function.

    Returns:
        None
    """
    # If there are not enough transitions stored in the memory buffer to form a minibatch, return without doing anything
    if len(memory) < BATCH_SIZE:
        return
    
    # Sample a minibatch of transitions from the memory buffer
    transitions = memory.sample(BATCH_SIZE)

    # Transpose the minibatch to create a batch of states, actions, rewards, and next_states
    batch = Transition(*zip(*transitions))

    # Compute a mask of non-final states and concatenate the batch elements
    non_final_mask = torch.tensor(tuple(map(lambda s: s is not None, batch.next_state)), device=device, dtype=torch.bool)
    non_final_next_states = torch.cat([s for s in batch.next_state if s is not None])
    state_batch = torch.cat(batch.state)
    action_batch = torch.cat(batch.action)
    reward_batch = torch.cat(batch.reward)

    # Compute Q(s_t, a) - the model computes Q(s_t), then we select the
    # columns of actions taken. These are the actions which would've been taken
    # for each batch state according to policy_net
    state_action_values = policy_net(state_batch).gather(1, action_batch) # current Q
    
    # Compute V(s_{t+1}) for all next states.
    # Expected values of actions for non_final_next_states are computed based
    # on the "older" target_net; selecting their best reward with max(1)[0].
    # This is merged based on the mask, such that we'll have either the expected
    # state value or 0 in case the state was final.
    next_state_values = torch.zeros(BATCH_SIZE, device=device)
    with torch.no_grad():
        next_state_values[non_final_mask] = target_net(non_final_next_states).max(1)[0]
    
    # Compute the expected Q values
    expected_state_action_values = (next_state_values * GAMMA) + reward_batch # r + dis * Q(s')
    # Compute the Huber loss between the current state-action values and the expected state-action values
    criterion = nn.SmoothL1Loss()
    loss = criterion(state_action_values, expected_state_action_values.unsqueeze(1))

    # Optimize the model
    optimizer.zero_grad()   # Set optimizer gradients
    loss.backward()
    # In-place gradient clipping
    torch.nn.utils.clip_grad_value_(policy_net.parameters(), 100)
    optimizer.step()

In [ ]:
start_time = time.time()
# Define the hyperparameters
BATCH_SIZE = 128
GAMMA = 0.99
TAU = 0.005
LR = 1e-4
eps_threshold = 1
# Get number of actions and state observations
n_actions = env.action_space.n
state, _ = env.reset()
n_observations = len(state)
# Initialize policy and target networks
policy_net = DQN(n_observations, n_actions).to(device)
target_net = DQN(n_observations, n_actions).to(device)
# Create an AdamW optimizer and pass the policy_net's parameters to it
optimizer = optim.AdamW(policy_net.parameters(), lr=LR, amsgrad=True)
# Initialize a replay memory with a capacity of 10000
memory = ReplayMemory(10000)

# Load the weights of the policy network to the target network
target_net.load_state_dict(policy_net.state_dict())

# Define the replay memory transition as a named tuple
Transition = namedtuple('Transition', ('state', 'action', 'next_state', 'reward'))
num_episodes = 500
episode_rewards = []
average_rewards = []
mean_rewards = []
optimal_action_percentages = []
for i_episode in range(num_episodes):
    # Initialize the environment and get its state
    state, _  = env.reset()
    
    state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    optimal_action_count = 0
    total_reward = 0
    for t in count():
        # Choose the next action using the current policy network
        action = select_action(state)

        # Check if the selected action is optimal
        with torch.no_grad():
            optimal_action = policy_net(state).max(1)[1].view(1, 1)
        if action == optimal_action:
            optimal_action_count += 1

        # Take the action and observe the next state, reward, and termination signal
        observation, reward, terminated, _, _ = env.step(action.item())
        total_reward += reward
        reward = torch.tensor([reward], device=device)
        done = terminated

        # Set the next state to None if the game is over, otherwise convert the observation to a tensor and unsqueeze it
        if terminated:
            next_state = None
        else:
            next_state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)

        # Store the transition in memory
        memory.push(state, action, next_state, reward)

        # Move to the next state
        state = next_state
        # Perform one step of the optimization on the policy network
        optimize_model()
        # Soft update of the target network's weights
        # θ′ ← τ θ + (1 −τ )θ′
        target_net_state_dict = target_net.state_dict()
        policy_net_state_dict = policy_net.state_dict()
        for key in policy_net_state_dict:
            target_net_state_dict[key] = policy_net_state_dict[key] * TAU + target_net_state_dict[key] * (1 - TAU)
        target_net.load_state_dict(target_net_state_dict)

        if done:
            break
    eps_threshold = eps_threshold - 2 / num_episodes if eps_threshold > 0.01 else 0.01
    # Calculate the average reward for this episode
   
    episode_rewards.append(total_reward)
    episode_rewards_array = np.array(episode_rewards)
    mean_last_20_episodes = np.mean(episode_rewards_array[-20:])
    mean_rewards.append(mean_last_20_episodes)
    if i_episode % 20 == 0 or i_episode<20:
        print(f"Episode {i_episode + 1}/{num_episodes}: episode reward {total_reward}, mean_last_20:{mean_last_20_episodes}, epsilon: {eps_threshold}")
# Generate x-values using np.arange or np.linspace
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Elapsed time: {elapsed_time} seconds")
mean_rewards_arr= np.array(mean_rewards)

arrname = "DQN_mean_rewards," + str(num_episodes) + "episodes.npy"
np.save(arrname,mean_rewards_arr)

plt.plot(mean_rewards_arr)

#plt.ylim(bottom=-200)
plt.xlabel('episode')
plt.ylabel('reward')
plt.title("DQN training mean rewards last 20 episodes")
figname = "DQN_training," + str(num_episodes) + "episodes.png"
plt.savefig(figname)

In [ ]:
def DQN_Select(state):
    state = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
            # t.max(1) will return the largest column value of each row.
            # the second column on max result is index of where max element was
            # found, so we pick action with the larger expected reward.
            return policy_net(state).max(1)[1].view(1, 1)

In [ ]:
env = gym.make("Acrobot-v1")
#Evaluate
# Set up the device to use (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy_net.to(device)
# Print the device being used
print(device)
ep_rewards = []
for episode in range(100):
    state, _ = env.reset()
    done = False
    ep_reward = 0
    while not done:
        action = DQN_Select(state)
        #energy +=
        # Step the environment
        state, reward, done, _ , _= env.step(action.item())
        ep_reward += reward
    ep_rewards.append(ep_reward)
    #print(f"episode: {episode}")

mean = sum(ep_rewards) / len(ep_rewards)
print(mean)
#plt.plot(ep_rewards)
x = range(len(ep_rewards))
plt.scatter(x,ep_rewards, marker = 'o')
plt.xlabel('Episode')
plt.ylabel('Reward')
plt.title('DQN evaluation, 500 episode rewards')
plt.savefig('DQN500_evaluate')
plt.show()
env.close()